# Multiple Regression based Model

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import numpy as np
from et_ts_generator import et_ts_list_regr
from models_plots import (
    plot_et_comparison,
    print_error_metrics,
)
from models_tools import (
    create_pixel_data_extrapolation_backward,
    create_pixel_data_extrapolation_forward,
    et_sd_list,
    find_directories,
    rad_list,
    var_list,
)
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


## General Principle:
We use a regression model based on the values of some variables from the previous (or next, for backward extrapolation) day and the current day to estimate evapotranspiration when no acquisition is available.

**Day-by-Day Forward Extrapolation** \
For a date without acquisition $t_i$:

$$
\text{RM}_{\text{forward}} =
\begin{cases}
\text{GPR}_{\text{forward}}, & \text{if } \text{SW}(t_i) > \epsilon \\
\text{DSLR}_{\text{forward}}, & \text{otherwise}
\end{cases}
$$

$$
\text{ET}(t_i) = \text{RM}_{\text{forward}} \Big(\text{Rad}_{sd}(t_i), \ \text{Rad}_{sd}(t_{i-1}), \ \text{ET}(t_{i-1}), \ \text{SW}(t_i)\Big)
$$

For a date with acquisition $t_k$:

$$
\text{ET}(t_k) = \text{ET}_{\text{obs}}(t_k)
$$

---

**Correction - Backward Extrapolation**

For each acquisition $t_k$ and for each previous value $t_j$ within the second half of the last extrapolated window (i.e., $t_j = t_k - j$ with $j = 1, 2, \dots, h$, where $h$ is half the length of the extrapolation period):

$$
\text{RM}_{\text{backward}} =
\begin{cases}
\text{GPR}_{\text{backward}}, & \text{if } \text{SW}(t_j) > \epsilon \\
\text{DSLR}_{\text{backward}}, & \text{otherwise}
\end{cases}
$$

$$
\text{ET}(t_j) = \text{RM}_{\text{backward}} \Big(\text{Rad}_{sd}(t_j), \ \text{Rad}_{sd}(t_{j+1}), \ \text{ET}(t_{j+1}), \ \text{SW}(t_j)\Big)
$$

**Legend:**

- **$GPR_{forward}$**: General Polynomial Regression model for forward extrapolation
- **$DSLR_{forward}$**: Dry-period-specific Linear Regression model for forward extrapolation
- **$GPR_{backward}$**: General Polynomial Regression model for backward extrapolation
- **$DSLR_{backward}$**: Dry-period-specific Linear Regression model for backward extrapolation
- **$SW$**: Volumetric Soil Water
- **$ET$:** daily evapotranspiration
- **$ET_{obs}$:** observed daily evapotranspiration (acquisition)
- **$Rad_{sd}$**: daily surface solar radiation  


**Zone-specific parameters:**

$\epsilon$: Threshold for switching from the general model the to dry-period specific model


**Regression models**

We selected the explanatory variables for the regression models using the Spearman correlation. Unlike a measure of the linear relationship between two variables, the Spearman correlation coefficient evaluates the degree to which an arbitrary monotonic function can describe the relationship between two variables.

As shown for the Beauce dataset, daily surface solar radiation and volumetric soil water content are the only two variables exhibiting a significant Spearman correlation with daily evapotranspiration (0.90 and -0.45, respectively).

Additionally, we included the previous day’s surface solar radiation and evapotranspiration (or the next day’s values for backward extrapolation) as explanatory variables, since our goal is to extrapolate data based on the support of a preceding (or subsequent) day.


### Training Dataset

To estimate the coefficients of the regression models, we created a dataset that captures the diversity of climate zones observable by the Meteosat satellite over Europe and Africa. For each zone, we selected a representative pixel and processed its data, then concatenated all the pixels to form a comprehensive dataset.

For zones exhibiting markedly different behaviors between wet and dry seasons, we split the dataset into two: one covering the entire year and another specific to the dry season.

We obtained the observed evapotranspiration, daily radiation, and daily soil moisture using the **prepare_et_single_date**, **prepare_daily_radiation**, and **prepare_daily_explanatory** functions from **ET-DATASET**, respectively. The data were then stored for each region of interest in a dedicated subdirectory within **SWEAT_TEST_DATA_PATH**.

In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

#### Data Beauce: European Temperate climate

In [ ]:
dir_beauce = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "beauce_EUT"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 404852
y = 5348897
dir_sd, dir_rad, dir_var = find_directories(dir_beauce)

Spearman correlations between daily evapotranspiration and other variables

In [ ]:
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)
rad = rad_list(dir_rad, min_date, max_date, x, y)
sw = var_list(dir_var, "sw", min_date, max_date, x, y)
tp = var_list(dir_var, "tp", min_date, max_date, x, y)
sro = var_list(dir_var, "sro", min_date, max_date, x, y)
src = var_list(dir_var, "src", min_date, max_date, x, y)


In [ ]:
spearmanr(et_sd, rad)[0].item()

In [ ]:
spearmanr(et_sd, sw)[0].item()

In [ ]:
spearmanr(et_sd, sro)[0].item()


In [ ]:
spearmanr(et_sd, src)[0].item()


In [ ]:
spearmanr(et_sd, tp)[0].item()


Forward Extrapolation

In [ ]:
x_beauce_for, y_beauce_for = create_pixel_data_extrapolation_forward(
    dir_beauce, min_date, max_date, x, y
)

Backward Extrapolation

In [ ]:
x_beauce_back, y_beauce_back = create_pixel_data_extrapolation_backward(
    dir_beauce, min_date, max_date, x, y
)


#### Data Merguellil: African Mediterranean climate

In [ ]:
dir_tunis = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "tunisia_MED"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 582777
y = 930444.0


Wet seasons

In [ ]:
x_tunis_for, y_tunis_for = create_pixel_data_extrapolation_forward(
    dir_tunis, "2024-01-01", "2024-12-31", x, y
)

In [ ]:
x_tunis_back, y_tunis_back = create_pixel_data_extrapolation_backward(
    dir_tunis, "2024-01-01", "2024-12-31", x, y
)


Dry season

In [ ]:
x_tunis_dry_back, y_tunis_dry_back = create_pixel_data_extrapolation_backward(
    dir_tunis, "2024-05-01", "2024-09-01", x, y
)

In [ ]:
x_tunis_dry_for, y_tunis_dry_for = create_pixel_data_extrapolation_forward(
    dir_tunis, "2024-06-01", "2024-09-01", x, y
)


#### Data Szeged: European Continental climate

In [ ]:
dir_szeged = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "szeged_EUC"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 464721
y = 5132982


In [ ]:
x_szeged_for, y_szeged_for = create_pixel_data_extrapolation_forward(
    dir_szeged, min_date, max_date, x, y
)

In [ ]:
x_szeged_back, y_szeged_back = create_pixel_data_extrapolation_backward(
    dir_szeged, min_date, max_date, x, y
)

#### Data Ouagadougou: African Sudanese climate

In [ ]:
dir_ouaga = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "ouagadougou_AFS"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 644483
y = 1334031


Whole year

In [ ]:
x_ouaga_for, y_ouaga_for = create_pixel_data_extrapolation_forward(
    dir_ouaga, "2024-01-01", "2024-12-31", x, y
)


In [ ]:
x_ouaga_back, y_ouaga_wet_back = create_pixel_data_extrapolation_backward(
    dir_ouaga, "2024-01-01", "2024-12-31", x, y
)


Dry season

In [ ]:
x_ouaga_dry1_for, y_ouaga_dry1_for = create_pixel_data_extrapolation_forward(
    dir_ouaga, "2024-01-01", "2024-05-01", x, y
)

In [ ]:
x_ouaga_dry1_back, y_ouaga_dry1_back = create_pixel_data_extrapolation_backward(
    dir_ouaga, "2024-01-01", "2024-05-01", x, y
)


In [ ]:
x_ouaga_dry2_for, y_ouaga_dry2_for = create_pixel_data_extrapolation_forward(
    dir_ouaga, "2024-11-01", "2024-12-31", x, y
)

In [ ]:
x_ouaga_dry2_back, y_ouaga_dry2_back = create_pixel_data_extrapolation_backward(
    dir_ouaga, "2024-11-01", "2024-12-31", x, y
)

#### Data Say: African Sahelian climate

In [ ]:
dir_say = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "say_SAH"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 424801
y = 1444966


Whole season

In [ ]:
x_say_wet_for, y_say_wet_for = create_pixel_data_extrapolation_forward(
    dir_say, "2024-01-01", "2024-12-31", x, y
)

In [ ]:
x_say_wet_back, y_say_wet_back = create_pixel_data_extrapolation_backward(
    dir_say, "2024-01-01", "2024-12-31", x, y
)

Dry season

In [ ]:
x_say_dry1_for, y_say_dry1_for = create_pixel_data_extrapolation_forward(
    dir_ouaga, "2024-01-01", "2024-07-01", x, y
)

In [ ]:
x_say_dry2_for, y_say_dry2_for = create_pixel_data_extrapolation_forward(
    dir_ouaga, "2024-10-15", "2024-12-31", x, y
)

In [ ]:
x_say_dry1_back, y_say_dry1_back = create_pixel_data_extrapolation_backward(
    dir_ouaga, "2024-01-01", "2024-07-01", x, y
)


In [ ]:
x_say_dry2_back, y_say_dry2_back = create_pixel_data_extrapolation_backward(
    dir_ouaga, "2024-10-15", "2024-12-31", x, y
)

#### Data Kikwit: African Equatorian climate

In [ ]:
dir_kikwit = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "kikwit_AEQ"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 251032
y = 9438668


In [ ]:
x_kikwit_for, y_kikwit_for = create_pixel_data_extrapolation_forward(
    dir_kikwit, "2024-01-01", "2024-12-31", x, y
)

In [ ]:
x_kikwit_back, y_kikwit_back = create_pixel_data_extrapolation_backward(
    dir_kikwit, "2024-01-01", "2024-12-31", x, y
)

In [ ]:
x_kikwit_dry_back, y_kikwit_dry_back = create_pixel_data_extrapolation_backward(
    dir_kikwit, "2024-05-01", "2024-09-01", x, y
)

In [ ]:
x_kikwit_dry_for, y_kikwit_dry_for = create_pixel_data_extrapolation_forward(
    dir_kikwit, "2024-05-01", "2024-09-01", x, y
)


#### Data Gaborone: Southern African climate

In [ ]:
dir_gaborone = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "gaborone_AAU"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 290968
y = 7344867


In [ ]:
x_gaborone_for, y_gaborone_for = create_pixel_data_extrapolation_forward(
    dir_gaborone, min_date, max_date, x, y
)

In [ ]:
x_gaborone_back, y_gaborone_back = create_pixel_data_extrapolation_backward(
    dir_gaborone, min_date, max_date, x, y
)


In [ ]:
x_gaborone_dry_for, y_gaborone_dry_for = (
    create_pixel_data_extrapolation_forward(
        dir_gaborone, "2024-05-01", "2024-10-15", x, y
    )
)


In [ ]:
x_gaborone_dry_back, y_gaborone_dry_back = (
    create_pixel_data_extrapolation_backward(
        dir_gaborone, "2024-05-01", "2024-10-15", x, y
    )
)


#### Data Concatenation

Forward Extrapolation

Whole year dataset

In [ ]:
X_wet_for = np.vstack(
    (
        x_beauce_for,
        x_tunis_for,
        x_ouaga_for,
        x_say_wet_for,
        x_kikwit_for,
        x_gaborone_for,
        x_szeged_for,
    )
)
Y_wet_for = np.hstack(
    (
        y_beauce_for,
        y_tunis_for,
        y_ouaga_for,
        y_say_wet_for,
        y_kikwit_for,
        y_gaborone_for,
        y_szeged_for,
    )
)

In [ ]:
print(X_wet_for.shape)
print(Y_wet_for.shape)

Dry season dataset

In [ ]:
X_dry_for = np.vstack(
    (
        x_tunis_dry_for,
        x_ouaga_dry1_for,
        x_ouaga_dry2_for,
        x_say_dry1_for,
        x_say_dry2_for,
        x_kikwit_dry_for,
        x_gaborone_dry_for,
    )
)
Y_dry_for = np.hstack(
    (
        y_tunis_dry_for,
        y_ouaga_dry1_for,
        y_ouaga_dry2_for,
        y_say_dry1_for,
        y_say_dry2_for,
        y_kikwit_dry_for,
        y_gaborone_dry_for,
    )
)

In [ ]:
print(X_dry_for.shape)
print(Y_dry_for.shape)

Backward Extrapolation

In [ ]:
X_wet_back = np.vstack(
    (
        x_beauce_back,
        x_tunis_back,
        x_ouaga_back,
        x_say_wet_back,
        x_kikwit_back,
        x_gaborone_back,
        x_szeged_back,
    )
)
Y_wet_back = np.hstack(
    (
        y_beauce_back,
        y_tunis_back,
        y_ouaga_wet_back,
        y_say_wet_back,
        y_kikwit_back,
        y_gaborone_back,
        y_szeged_back,
    )
)


In [ ]:
X_dry_back = np.vstack(
    (
        x_tunis_dry_back,
        x_ouaga_dry1_back,
        x_ouaga_dry2_back,
        x_say_dry1_back,
        x_say_dry2_back,
        x_kikwit_dry_back,
        x_gaborone_dry_back,
    )
)
Y_dry_back = np.hstack(
    (
        y_tunis_dry_back,
        y_ouaga_dry1_back,
        y_ouaga_dry2_back,
        y_say_dry1_back,
        y_say_dry2_back,
        y_kikwit_dry_back,
        y_gaborone_dry_back,
    )
)


### Regression models

#### Forward Extrapolation

Whole year model

We found that the best regression model for day-by-day forward extrapolation in the general case (excluding dry season conditions) is a polynomial Ridge Regression.
Adding second-order terms (squared variables and interaction terms) allows the model to capture non-linear relationships. The constraint on the coefficient magnitudes imposed by Ridge helps to limit overfitting, introducing a slight bias in exchange for reduced variance. We also took care to standardize the variables to account for differences in scale, which is important because Ridge penalizes the norm of the coefficients.

To determine the optimal alpha (regularization parameter), we performed cross-validation over 5 splits of the dataset, while respecting the temporal order of the data.

In [ ]:
alphas = np.logspace(-3, 3, 50)
tscv = TimeSeriesSplit(n_splits=15)

model_wet_for = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        (
            "ridge",
            RidgeCV(
                alphas=alphas, cv=tscv, scoring="neg_root_mean_squared_error"
            ),
        ),
    ]
)


model_wet_for.fit(X_wet_for, Y_wet_for)
best_alpha = model_wet_for.named_steps["ridge"].alpha_
coefficients = model_wet_for.named_steps["ridge"].coef_

print("Best alpha: ", best_alpha)
print("Number of coefficients :", len(coefficients))


In [ ]:
model_wet_for.score(X_wet_for, Y_wet_for)

Dry season

We found that the best regression model for day-by-day forward extrapolation in the dry season conditions is a Multiple Linear Regression.

In [ ]:
model_dry_for = LinearRegression()
model_dry_for.fit(X_dry_for, Y_dry_for)

In [ ]:
model_dry_for.score(X_dry_for, Y_dry_for)

#### Backward Extrapolation

We found that the best regression model for day-by-day backward extrapolation in the general case (excluding dry season conditions) is a polynomial Ridge Regression.

In [ ]:
alphas = np.logspace(-3, 3, 50)
tscv = TimeSeriesSplit(n_splits=15)

model_wet_back = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        (
            "ridge",
            RidgeCV(
                alphas=alphas, cv=tscv, scoring="neg_root_mean_squared_error"
            ),
        ),
    ]
)
model_wet_back.fit(X_wet_back, Y_wet_back)

best_alpha = model_wet_back.named_steps["ridge"].alpha_
coefficients = model_wet_back.named_steps["ridge"].coef_
print("Best alpha :", best_alpha)
print("Number of coefficients :", len(coefficients))


In [ ]:
model_wet_back.score(X_wet_back, Y_wet_back)

We found that the best regression model for day-by-day backward extrapolation in the dry season conditions is a Multiple Linear Regression.

In [ ]:
model_dry_back = LinearRegression()
model_dry_back.fit(X_dry_back, Y_dry_back)


In [ ]:
model_dry_back.score(X_dry_back, Y_dry_back)


### Test

 #### Niakhar - Central Senegal

In [ ]:
dir_niakhar = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "niakhar"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 345700
y = 1590936
dir_sd, _, _ = find_directories(dir_niakhar)
et_ts = et_ts_list_regr(
    dir_niakhar,
    min_date,
    max_date,
    model_wet_for,
    model_dry_for,
    model_wet_back,
    model_dry_back,
    0.15,
    3,
    x,
    y,
)
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)

In [ ]:
plot_et_comparison(et_sd, et_ts, "2024-01-01", "2024-12-31")

In [ ]:
print_error_metrics(et_sd, et_ts, 3)

#### Aurade - South-West France

In [ ]:
dir_aurade = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "aurade"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 349095
y = 4830273
dir_sd, _, _ = find_directories(dir_aurade)
et_ts = et_ts_list_regr(
    dir_aurade,
    min_date,
    max_date,
    model_wet_for,
    model_dry_for,
    model_wet_back,
    model_dry_back,
    0.15,
    3,
    x,
    y,
)
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)


In [ ]:
plot_et_comparison(et_sd, et_ts, min_date, max_date)

In [ ]:
print_error_metrics(et_sd, et_ts, 3)


#### Polokwa - South-Africa

In [ ]:
dir_polokwa = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "polokwa"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 758000.0
y = 7399056
dir_sd, _, _ = find_directories(dir_polokwa)
et_ts = et_ts_list_regr(
    dir_polokwa,
    min_date,
    max_date,
    model_wet_for,
    model_dry_for,
    model_wet_back,
    model_dry_back,
    0.15,
    3,
    x,
    y,
)
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)


In [ ]:
plot_et_comparison(et_sd, et_ts, min_date, max_date)


In [ ]:
print_error_metrics(et_sd, et_ts, 3)


#### Andalousie - Southern Spain

In [ ]:
dir_anda = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "andalousie"
)
min_date = "2024-01-01"
max_date = "2024-12-31"
x = 261520
y = 4163901
dir_sd, _, _ = find_directories(dir_anda)
et_ts = et_ts_list_regr(
    dir_anda,
    min_date,
    max_date,
    model_wet_for,
    model_dry_for,
    model_wet_back,
    model_dry_back,
    0.25,
    3,
    x,
    y,
)
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)


In [ ]:
plot_et_comparison(et_sd, et_ts, min_date, max_date)


In [ ]:
print_error_metrics(et_sd, et_ts, 3)


#### Champagne - Eastern France

In [ ]:
dir_champ = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "champagne"
)
x = 580758
y = 5420539
dir_sd, _, _ = find_directories(dir_champ)
et_ts = et_ts_list_regr(
    dir_champ,
    min_date,
    max_date,
    model_wet_for,
    model_dry_for,
    model_wet_back,
    model_dry_back,
    0.15,
    3,
    x,
    y,
)
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)


In [ ]:
plot_et_comparison(et_sd, et_ts, min_date, max_date)

In [ ]:
print_error_metrics(et_sd, et_ts, 5)


#### Kisangani - RDC

In [ ]:
dir_kisan = os.path.join(
    os.environ.get("SWEAT_TEST_DATA_PATH"), "data_validation", "kisangani"
)
x = 286332
y = 50297
dir_sd, _, _ = find_directories(dir_kisan)
et_ts = et_ts_list_regr(
    dir_kisan,
    min_date,
    max_date,
    model_wet_for,
    model_dry_for,
    model_wet_back,
    model_dry_back,
    0.15,
    3,
    x,
    y,
)
et_sd = et_sd_list(dir_sd, min_date, max_date, x, y)


In [ ]:
plot_et_comparison(et_sd, et_ts, min_date, max_date)

In [ ]:
print_error_metrics(et_sd, et_ts, 3)